In [29]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Subset
import random
from collections import defaultdict
from collections import Counter
from sklearn.decomposition import PCA

Import all necessary libraries

In [30]:
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

CAT_LABEL = 3
DOG_LABEL = 5

# Load dataset
total_trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

def balanced_cat_dog_indices(dataset, total_samples):
    class_indices = defaultdict(list)

    for i, (_, label) in enumerate(dataset):
        if label in [CAT_LABEL, DOG_LABEL]:
            class_indices[label].append(i)

    samples_per_class = total_samples // 2

    cat_indices = random.sample(class_indices[CAT_LABEL], samples_per_class)
    dog_indices = random.sample(class_indices[DOG_LABEL], samples_per_class)

    indices = cat_indices + dog_indices
    random.shuffle(indices)
    return indices

indices_50 = balanced_cat_dog_indices(total_trainset, 50)
indices_500 = balanced_cat_dog_indices(total_trainset, 500)

trainset_50 = Subset(total_trainset, indices_50)
trainset_500 = Subset(total_trainset, indices_500)


class CatDogDataset(torch.utils.data.Dataset):
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        label = 0 if label == CAT_LABEL else 1
        return image, label

trainset_50 = CatDogDataset(trainset_50)
trainset_500 = CatDogDataset(trainset_500)

trainloader_50 = torch.utils.data.DataLoader(
    trainset_50,
    batch_size=8,
    shuffle=True
)

trainloader_500 = torch.utils.data.DataLoader(
    trainset_500,
    batch_size=32,
    shuffle=True
)

full_testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

test_indices = [
    i for i, (_, label) in enumerate(full_testset)
    if label in [CAT_LABEL, DOG_LABEL]
]

testset = CatDogDataset(Subset(full_testset, test_indices))

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)


labels_50 = [label for _, label in trainset_50]
labels_500 = [label for _, label in trainset_500]

print("50-sample distribution:", Counter(labels_50))
print("500-sample distribution:", Counter(labels_500))



50-sample distribution: Counter({0: 25, 1: 25})
500-sample distribution: Counter({0: 250, 1: 250})


In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Load the pretrained model for each one
model50 = models.resnet18(pretrained=True)
model500 = models.resnet18(pretrained=True)

model50.fc = nn.Linear(model50.fc.in_features, 2)
model500.fc = nn.Linear(model500.fc.in_features, 2)

for name, param in model500.named_parameters():
    if ('layer3' in name) or ('layer4' in name) or ('fc' in name):
        param.requires_grad = True
    else:
        param.requires_grad = False

for name, param in model500.named_parameters():
    if ('layer3' in name) or ('layer4' in name) or ('fc' in name):
        param.requires_grad = True
    else:
        param.requires_grad = False


#define loss function and optimizer
loss_function = nn.CrossEntropyLoss()

optimizer50 = optim.Adam(
    filter(lambda p: p.requires_grad, model50.parameters()),
    lr=1e-4
)

optimizer500 = optim.Adam(
    filter(lambda p: p.requires_grad, model500.parameters()),
    lr=1e-4
)

print('ready for training!')

D:\Arjun\Anaconda\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\Arjun\Anaconda\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ready for training!


Train the final layers of each!

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    model.train()

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        if batch % 10 == 0:
            loss_val = loss.item()
            current = batch * len(X)
            print(f"loss: {loss_val:.4f}  [{current}/{size}]")

            
def test_loop(dataloader, model, loss_fn, device):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    test_loss, correct = 0.0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)

            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(dim=1) == y).sum().item()

    test_loss /= num_batches
    accuracy = correct / size

    print(f"Test Error:\n Accuracy: {accuracy*100:.2f}%, Avg loss: {test_loss:.4f}\n")

num_epochs = 10

print("Training model with 50 samples")
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}")
    train_loop(trainloader_50, model50, loss_function, optimizer50, device)
    test_loop(testloader, model50, loss_function, device)

print("Training model with 500 samples")
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}")
    train_loop(trainloader_500, model500, loss_function, optimizer500, device)
    test_loop(testloader, model500, loss_function, device)


Training model with 50 samples
Epoch 1
loss: 0.6264  [0/50]
Test Error:
 Accuracy: 53.10%, Avg loss: 0.7740

Epoch 2
loss: 0.4678  [0/50]


In [ ]:
activations50 = []
activations500 = []

act50_labels = []
act500_labels = []


def hook50_fn(module, input, output):
    activations50.append(output.detach().cpu())

def hook500_fn(module, input, output):
    activations500.append(output.detach().cpu())


hook50 = model50.avgpool.register_forward_hook(hook50_fn)
hook500 = model500.avgpool.register_forward_hook(hook500_fn)

model50.eval()
model500.eval()

with torch.no_grad():
    for X, y in testloader:
        X = X.to(device)
        _ = model50(X)          # forward pass triggers hook
        act50_labels.extend(y.numpy())

with torch.no_grad():
    for X, y in testloader:
        X = X.to(device)
        _ = model500(X)          # forward pass triggers hook
        act500_labels.extend(y.numpy())

hook50.remove()
hook500.remove()

acts50 = torch.cat(activations50, dim=0)   # [N, 512, 1, 1]
acts50 = acts50.view(acts50.size(0), -1)     # [N, 512]

acts500 = torch.cat(activations500, dim=0)   # [N, 512, 1, 1]
acts500 = acts500.view(acts500.size(0), -1)     # [N, 512]


acts_all = torch.cat([acts50, acts500], dim=0)

pca = PCA(n_components=2)
acts_all_2d = pca.fit_transform(acts_all.numpy())

n50 = acts50.shape[0]

acts50_2d = acts_all_2d[:n50]
acts500_2d = acts_all_2d[n50:]
labels50 = np.array(act50_labels)
labels500 = np.array(act500_labels)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

# model50
axes[0].scatter(
    acts50_2d[labels50 == 0, 0],
    acts50_2d[labels50 == 0, 1],
    label="Cat",
    alpha=0.6
)
axes[0].scatter(
    acts50_2d[labels50 == 1, 0],
    acts50_2d[labels50 == 1, 1],
    label="Dog",
    alpha=0.6
)
axes[0].set_title("Model trained on 50 samples")
axes[0].legend()

# model 500
axes[1].scatter(
    acts500_2d[labels500 == 0, 0],
    acts500_2d[labels500 == 0, 1],
    label="Cat",
    alpha=0.6
)
axes[1].scatter(
    acts500_2d[labels500 == 1, 0],
    acts500_2d[labels500 == 1, 1],
    label="Dog",
    alpha=0.6
)
axes[1].set_title("Model trained on 500 samples")
axes[1].legend()

plt.suptitle("PCA of ResNet AvgPool Activations")
plt.show()
